# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [ ]:
! pip install -q schedule pytest # установка библиотек, если ещё не

In [ ]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import json
import re
import time
from urllib.parse import urljoin

import requests
import schedule
from bs4 import BeautifulSoup

BASE_URL = 'http://books.toscrape.com/catalogue/'
RATING_MAP = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}

## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [ ]:
def get_book_data(book_url: str, session=None) -> dict:
    """
    Собирает данные об одной книге со страницы сайта Books to Scrape.

    Args:
        book_url (str): полный URL страницы книги.
        session (requests.Session | None): сессия для повторного
            использования соединения. Если None, используется
            обычный requests.get.

    Returns:
        dict: словарь с данными книги:
            title (str) - название;
            price (float) - цена в фунтах;
            rating (int) - рейтинг от 1 до 5;
            in_stock (int) - количество экземпляров в наличии;
            category (str) - категория (жанр);
            description (str) - описание (пустая строка, если его нет);
            product_info (dict) - таблица Product Information.

    Raises:
        requests.HTTPError: если сервер вернул код ошибки.
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    getter = session or requests
    response = getter.get(book_url, timeout=10)
    response.raise_for_status()
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, 'html.parser')

    main = soup.find('div', class_='product_main')
    title = main.find('h1').text.strip()

    price_text = main.find('p', class_='price_color').text.strip()
    price = float(price_text.lstrip('£'))

    rating_word = main.find('p', class_='star-rating')['class'][1]
    rating = RATING_MAP.get(rating_word, 0)

    availability = main.find('p', class_='availability').text.strip()
    match = re.search(r'\d+', availability)
    in_stock = int(match.group()) if match else 0

    desc_header = soup.find('div', id='product_description')
    description = (desc_header.find_next_sibling('p').text.strip()
                   if desc_header else '')

    breadcrumb = soup.find('ul', class_='breadcrumb').find_all('li')
    category = breadcrumb[2].text.strip()

    table = soup.find('table', class_='table-striped')
    product_info = {row.th.text.strip(): row.td.text.strip()
                    for row in table.find_all('tr')}

    return {
        'title': title,
        'price': price,
        'rating': rating,
        'in_stock': in_stock,
        'category': category,
        'description': description,
        'product_info': product_info,
    }
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [ ]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'title': 'A Light in the Attic',
 'price': 51.77,
 'rating': 3,
 'in_stock': 22,
 'category': 'Poetry',
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [ ]:
def scrape_books(is_save: bool = False,
                 file_path: str = 'books_data.txt',
                 max_pages: int | None = None)  -> list:
    """
     Собирает данные обо всех книгах каталога Books to Scrape.

    Проходит по страницам каталога вида
    http://books.toscrape.com/catalogue/page-{N}.html и для каждой
    книги вызывает get_book_data.

    Args:
        is_save (bool): если True, результат сохраняется в файл
            (одна книга в формате JSON на строку).
        file_path (str): путь к файлу для сохранения.
        max_pages (int | None): максимальное число страниц для обхода;
            None означает все страницы.

    Returns:
        list[dict]: список словарей с данными книг
            (структура как в get_book_data).
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    books = []
    page = 1
    with requests.Session() as session:
        while max_pages is None or page <= max_pages:
            page_url = f'{BASE_URL}page-{page}.html'
            response = session.get(page_url, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            for article in soup.find_all('article', class_='product_pod'):
                book_url = urljoin(page_url, article.h3.a['href'])
                books.append(get_book_data(book_url, session))
            print(f'Страница {page} готова, всего книг: {len(books)}')

            if soup.find('li', class_='next') is None:
                break
            page += 1

    if is_save:
        with open(file_path, 'w', encoding='utf-8') as file:
            for book in books:
                file.write(json.dumps(book, ensure_ascii=False) + '\n')
    return books
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [ ]:
# Проверка работоспособности функции
res = scrape_books(is_save=True) # Допишите ваши аргументы
print(type(res), len(res)) # и проверки

Страница 1 готова, всего книг: 20
Страница 2 готова, всего книг: 40
Страница 3 готова, всего книг: 60
Страница 4 готова, всего книг: 80
Страница 5 готова, всего книг: 100
Страница 6 готова, всего книг: 120
Страница 7 готова, всего книг: 140
Страница 8 готова, всего книг: 160
Страница 9 готова, всего книг: 180
Страница 10 готова, всего книг: 200
Страница 11 готова, всего книг: 220
Страница 12 готова, всего книг: 240
Страница 13 готова, всего книг: 260
Страница 14 готова, всего книг: 280
Страница 15 готова, всего книг: 300
Страница 16 готова, всего книг: 320
Страница 17 готова, всего книг: 340
Страница 18 готова, всего книг: 360
Страница 19 готова, всего книг: 380
Страница 20 готова, всего книг: 400
Страница 21 готова, всего книг: 420
Страница 22 готова, всего книг: 440
Страница 23 готова, всего книг: 460
Страница 24 готова, всего книг: 480
Страница 25 готова, всего книг: 500
Страница 26 готова, всего книг: 520
Страница 27 готова, всего книг: 540
Страница 28 готова, всего книг: 560
Стран

## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [ ]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ
def job():
    """Запускает сбор данных и сохраняет результат в файл."""
    print(f'[{time.strftime("%Y-%m-%d %H:%M:%S")}] Старт выгрузки')
    books = scrape_books(is_save=True)
    print(f'Готово, сохранено книг: {len(books)}')


schedule.every().day.at('19:00').do(job)

while True:
    schedule.run_pending()
    time.sleep(60)
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

KeyboardInterrupt: 

## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [ ]:
! mkdir -p tests

In [ ]:
%%writefile pytest.ini
[pytest]
pythonpath = .

Writing pytest.ini


In [ ]:
%%writefile scraper.py
"""Парсер каталога книг с сайта Books to Scrape."""

import json
import re
import time
from urllib.parse import urljoin

import requests
import schedule
from bs4 import BeautifulSoup

BASE_URL = 'http://books.toscrape.com/catalogue/'
RATING_MAP = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}


def get_book_data(book_url: str, session=None) -> dict:
    """
    Собирает данные об одной книге со страницы сайта Books to Scrape.

    Args:
        book_url (str): полный URL страницы книги.
        session (requests.Session | None): сессия для повторного
            использования соединения. Если None, используется
            обычный requests.get.

    Returns:
        dict: словарь с данными книги: title, price, rating, in_stock,
            category, description, product_info.

    Raises:
        requests.HTTPError: если сервер вернул код ошибки.
    """
    getter = session or requests
    response = getter.get(book_url, timeout=10)
    response.raise_for_status()
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, 'html.parser')

    main = soup.find('div', class_='product_main')
    title = main.find('h1').text.strip()

    price_text = main.find('p', class_='price_color').text.strip()
    price = float(price_text.lstrip('£'))

    rating_word = main.find('p', class_='star-rating')['class'][1]
    rating = RATING_MAP.get(rating_word, 0)

    availability = main.find('p', class_='availability').text.strip()
    match = re.search(r'\d+', availability)
    in_stock = int(match.group()) if match else 0

    desc_header = soup.find('div', id='product_description')
    description = (desc_header.find_next_sibling('p').text.strip()
                   if desc_header else '')

    breadcrumb = soup.find('ul', class_='breadcrumb').find_all('li')
    category = breadcrumb[2].text.strip()

    table = soup.find('table', class_='table-striped')
    product_info = {row.th.text.strip(): row.td.text.strip()
                    for row in table.find_all('tr')}

    return {
        'title': title,
        'price': price,
        'rating': rating,
        'in_stock': in_stock,
        'category': category,
        'description': description,
        'product_info': product_info,
    }


def scrape_books(is_save: bool = False,
                 file_path: str = 'books_data.txt',
                 max_pages: int | None = None) -> list:
    """
    Собирает данные обо всех книгах каталога Books to Scrape.

    Args:
        is_save (bool): если True, результат сохраняется в файл
            (одна книга в формате JSON на строку).
        file_path (str): путь к файлу для сохранения.
        max_pages (int | None): максимальное число страниц;
            None означает все страницы.

    Returns:
        list[dict]: список словарей с данными книг.
    """
    books = []
    page = 1
    with requests.Session() as session:
        while max_pages is None or page <= max_pages:
            page_url = f'{BASE_URL}page-{page}.html'
            response = session.get(page_url, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            for article in soup.find_all('article', class_='product_pod'):
                book_url = urljoin(page_url, article.h3.a['href'])
                books.append(get_book_data(book_url, session))
            print(f'Страница {page} готова, всего книг: {len(books)}')

            if soup.find('li', class_='next') is None:
                break
            page += 1

    if is_save:
        with open(file_path, 'w', encoding='utf-8') as file:
            for book in books:
                file.write(json.dumps(book, ensure_ascii=False) + '\n')
    return books


def job():
    """Запускает сбор данных и сохраняет результат в файл."""
    print(f'[{time.strftime("%Y-%m-%d %H:%M:%S")}] Старт выгрузки')
    books = scrape_books(is_save=True,
                         file_path='artifacts/books_data.txt')
    print(f'Готово, сохранено книг: {len(books)}')


if __name__ == '__main__':
    schedule.every().day.at('19:00', 'Asia/Omsk').do(job)
    while True:
        schedule.run_pending()
        time.sleep(60)

Writing scraper.py


In [ ]:
%%writefile tests/test_scraper.py
import pytest

from scraper import get_book_data, scrape_books

BOOK_URL = ('http://books.toscrape.com/catalogue/'
            'a-light-in-the-attic_1000/index.html')
EXPECTED_KEYS = {'title', 'price', 'rating', 'in_stock',
                 'category', 'description', 'product_info'}


@pytest.fixture(scope='module')
def book():
    """Загружает одну книгу один раз для всех тестов модуля."""
    return get_book_data(BOOK_URL)


def test_book_is_dict_with_keys(book):
    assert isinstance(book, dict)
    assert EXPECTED_KEYS.issubset(book.keys())


def test_book_field_values(book):
    assert book['title'] == 'A Light in the Attic'
    assert book['price'] == 51.77
    assert book['category'] == 'Poetry'
    assert book['product_info']['UPC'] == 'a897fe39b1053632'


def test_book_rating_in_range(book):
    assert 1 <= book['rating'] <= 5


def test_scrape_one_page_returns_20_books():
    books = scrape_books(max_pages=1)
    assert len(books) == 20


def test_scrape_saves_file(tmp_path):
    file_path = tmp_path / 'books.txt'
    scrape_books(is_save=True, file_path=file_path, max_pages=1)
    lines = file_path.read_text(encoding='utf-8').splitlines()
    assert len(lines) == 20

Writing tests/test_scraper.py


In [ ]:
# Ячейка для демонстрации работоспособности
# Сам код находится в отдельном скрипте tests/test_scraper.py
! pytest tests/test_scraper.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
configfile: pytest.ini
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collected 5 items                                                              

tests/test_scraper.py::test_book_is_dict_with_keys PASSED                [ 20%]
tests/test_scraper.py::test_book_field_values PASSED                     [ 40%]
tests/test_scraper.py::test_book_rating_in_range PASSED                  [ 60%]
tests/test_scraper.py::test_scrape_one_page_returns_20_books PASSED      [ 80%]
tests/test_scraper.py::test_scrape_saves_file PASSED                     [100%]

============================== 5 passed in 5.74s ===============================


## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: необходимо создать программу автоматического сбора данных из сайта, извлечения информации о книгах, автоматическим запуском и записи результата. Вроде получилось`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: Застряла на работе с GIT не с первого раза получалось писать в нем код для всего объедениния и записи в README`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: пишу на интерпритаторе Анаконда, использовала в части как писать в ней. ПРОМТ + ОТВЕТ
питон у меня установлена анаконда

С Анакондой отдельно ставить Python не нужно, он в ней уже есть.

Сначала проверьте, видит ли Git Bash Python

В Git Bash:

bash
python --version
Если появилось command not found или открылся Microsoft Store, значит, Git Bash не видит Python из Анаконды. Делайте в Anaconda Prompt, как описано ниже.

Пуск → найдите Anaconda Prompt → откройте. В начале строки будет (base).
Перейдите в папку проекта. Путь удобно скопировать из адресной строки Проводника, когда вы находитесь внутри books_scraper:
cd /d C:\Users\ВашеИмя\Documents\books_scraper

Флаг /d нужен, если папка лежит на другом диске, например на D:.
3. Создайте и включите окружение. Здесь синтаксис Windows: обратные слэши и без source:

python -m venv venv
venv\Scripts\activate

В начале строки появится (venv).
4. Установите библиотеки и сохраните список:

pip install requests beautifulsoup4 schedule pytest pytz
pip freeze > requirements.txt
type requirements.txt

type здесь работает как cat: показывает содержимое файла. Должно быть 10–15 строк.
5. Проверьте тесты:

pytest -v

Должно быть 5 passed.

После этого возвращайтесь в Git Bash и продолжайте README. Git-команды в Anaconda Prompt выполнять не нужно, там Git может быть не виден.

Почему не conda create

У Анаконды есть свои окружения (conda create), но для этого задания лучше обычный venv. В conda-окружении pip freeze часто записывает строки вида requests @ file:///C:/... с путями вашего компьютера, и на другом компьютере такой requirements.txt не установится. venv, созданный из Python Анаконды, даёт чистый список.`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: в целом все понятно. Но попробую еще прогнать несколько сайтов чтобы закрепить результат`

**5. Самооценка по критериям задания**:

`Ответ: пока сложно, но уже лучше`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: ...`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

